In [1]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces
from collections import deque
import random

In [ ]:
class SnakeEnv(gym.Env):
    """
    Entorno personalizado de Snake con obstáculos dinámicos y comida mala.
    Formulado como MDP episódico y parcialmente estocástico.
    
    Espacio de estados: vector de 14 elementos binarios/numéricos (discretizable)
    Espacio de acciones: Discrete(3) -> [izquierda, recto, derecha]
    """

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 10}

    # Direcciones: (fila, columna)
    DIRECTIONS = [
        (-1, 0),  # 0: Arriba
        (0,  1),  # 1: Derecha
        (1,  0),  # 2: Abajo
        (0, -1),  # 3: Izquierda
    ]

    def __init__(self, grid_size=10, obstacle_move_freq=5, max_steps=500):
        super().__init__()
        self.grid_size = grid_size
        self.obstacle_move_freq = obstacle_move_freq  # cada cuántos pasos se mueven los obstáculos
        self.max_steps = max_steps
        self.n_obstacles = 3

        # ---------------------------------------------------------------
        # Espacio de ACCIONES: Discrete(3)
        # 0 = girar izquierda, 1 = seguir recto, 2 = girar derecha
        # Acciones relativas a la dirección actual (más fácil de aprender
        # que acciones absolutas Norte/Sur/Este/Oeste)
        # ---------------------------------------------------------------
        self.action_space = spaces.Discrete(3)

        # ---------------------------------------------------------------
        # Espacio de OBSERVACIONES: vector de 14 valores
        # [0-2]  peligro_delante, peligro_izquierda, peligro_derecha
        # [3-6]  dirección actual (one-hot: arriba, derecha, abajo, izquierda)
        # [7]    comida_buena_arriba, [8] comida_buena_derecha,
        # [9]    comida_buena_abajo,  [10] comida_buena_izquierda
        # [11]   comida_mala_delante
        # [12]   obstaculo_delante
        # [13]   longitud normalizada
        # ---------------------------------------------------------------
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(14,), dtype=np.float32
        )

        # Estado interno (se inicializa en reset)
        self.snake = None
        self.direction = None
        self.good_food = None
        self.bad_food = None
        self.obstacles = None
        self.steps = 0

    # ------------------------------------------------------------------
    # reset(): Inicia un nuevo episodio — devuelve observación inicial
    # ------------------------------------------------------------------
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        mid = self.grid_size // 2
        # La serpiente empieza con 3 celdas en el centro mirando a la derecha
        self.snake = deque([
            (mid, mid),
            (mid, mid - 1),
            (mid, mid - 2),
        ])
        self.direction = 1  # mirando a la derecha

        self.steps = 0
        self._place_food()
        self._place_obstacles()

        obs = self._get_observation()
        info = {}
        return obs, info

    # ------------------------------------------------------------------
    # step(action): Núcleo del MDP — aplica la acción y devuelve transición
    # Devuelve: (obs, reward, terminated, truncated, info)
    # ------------------------------------------------------------------
    def step(self, action):
        self.steps += 1

        # --- Actualizar dirección (acción relativa) ---
        # 0=izquierda, 1=recto, 2=derecha respecto a la dirección actual
        if action == 0:
            self.direction = (self.direction - 1) % 4  # giro a la izquierda
        elif action == 2:
            self.direction = (self.direction + 1) % 4  # giro a la derecha
        # action == 1: seguir recto, no cambia la dirección

        # --- Mover la cabeza ---
        dr, dc = self.DIRECTIONS[self.direction]
        head_r, head_c = self.snake[0]
        new_head = (head_r + dr, head_c + dc)

        # --- Calcular recompensa y condición de fin ---
        # Función de recompensa diseñada para evitar reward hacking:
        # El paso negativo (-0.01) evita que la serpiente simplemente
        # sobreviva sin comer; las recompensas de distancia guían el aprendizaje
        # inicial cuando los eventos de comida son escasos.
        reward = -0.01  # penalización por paso (fomenta eficiencia)
        terminated = False
        info = {"cause": None}

        # Comprobar colisión con pared o cuerpo (fin de episodio)
        if self._is_collision(new_head):
            reward = -20.0
            terminated = True
            info["cause"] = "collision"
            obs = self._get_observation()
            return obs, reward, terminated, False, info

        # Mover la serpiente: añadimos cabeza, quitamos cola (salvo que coma)
        self.snake.appendleft(new_head)

        if new_head == self.good_food:
            # Come comida buena: crece (no quitamos cola)
            reward += 10.0
            self._place_food()  # nueva comida buena
        elif new_head == self.bad_food:
            # Come comida mala: muere (fin de episodio)
            reward = -10.0
            terminated = True
            info["cause"] = "bad_food"
            obs = self._get_observation()
            return obs, reward, terminated, False, info
        else:
            self.snake.pop()  # movimiento normal: quitar cola
            # Recompensa de distancia: acercarse a comida buena es positivo
            reward += self._distance_reward(new_head)

        # Verificar colisión con obstáculo después del movimiento
        if new_head in self.obstacles:
            reward = -20.0
            terminated = True
            info["cause"] = "obstacle"
            obs = self._get_observation()
            return obs, reward, terminated, False, info

        # --- Mover obstáculos periódicamente (estocasticidad del entorno) ---
        # Esto conecta con la dinámica estocástica del MDP:
        # P(s'|s,a) no es completamente determinista
        if self.steps % self.obstacle_move_freq == 0:
            self._move_obstacles()

        # Truncamiento: máximo de pasos por episodio
        truncated = self.steps >= self.max_steps

        obs = self._get_observation()
        return obs, reward, truncated, truncated, info

    # ------------------------------------------------------------------
    # render(): Representación visual del entorno
    # ------------------------------------------------------------------
    def render(self):
        grid = [['.' for _ in range(self.grid_size)] for _ in range(self.grid_size)]
        for r, c in list(self.snake)[1:]:
            if 0 <= r < self.grid_size and 0 <= c < self.grid_size:
                grid[r][c] = 'o'
        hr, hc = self.snake[0]
        grid[hr][hc] = 'H'
        if self.good_food:
            grid[self.good_food[0]][self.good_food[1]] = 'F'
        if self.bad_food:
            grid[self.bad_food[0]][self.bad_food[1]] = 'X'
        for (r, c) in self.obstacles:
            grid[r][c] = '#'

        print(f"\nPaso: {self.steps} | Longitud: {len(self.snake)}")
        print('+' + '-' * self.grid_size + '+')
        for row in grid:
            print('|' + ''.join(row) + '|')
        print('+' + '-' * self.grid_size + '+')

    def close(self):
        pass

    # ------------------------------------------------------------------
    # Métodos auxiliares privados
    # ------------------------------------------------------------------

    def _get_observation(self) -> np.ndarray:
        """
        Construye el vector de estado (observación).
        Usa posiciones RELATIVAS a la dirección actual para que el estado
        sea invariante a la orientación — esto reduce el espacio efectivo
        y facilita el aprendizaje tanto en Q-learning como en DQN.
        """
        head = self.snake[0]
        dir_idx = self.direction

        # Direcciones relativas (delante, izquierda, derecha respecto a la actual)
        front = self.DIRECTIONS[dir_idx]
        left  = self.DIRECTIONS[(dir_idx - 1) % 4]
        right = self.DIRECTIONS[(dir_idx + 1) % 4]

        def next_cell(pos, d):
            return (pos[0] + d[0], pos[1] + d[1])

        # [0-2] Peligro en las tres direcciones relativas
        danger_front = float(self._is_collision(next_cell(head, front)) or
                             next_cell(head, front) in self.obstacles)
        danger_left  = float(self._is_collision(next_cell(head, left))  or
                             next_cell(head, left)  in self.obstacles)
        danger_right = float(self._is_collision(next_cell(head, right)) or
                             next_cell(head, right) in self.obstacles)

        # [3-6] Dirección actual en one-hot (arriba=0, derecha=1, abajo=2, izq=3)
        dir_one_hot = [float(dir_idx == i) for i in range(4)]

        # [7-10] Posición relativa de la comida buena (norte/sur/este/oeste)
        gf_r, gf_c = self.good_food
        h_r, h_c   = head
        food_up    = float(gf_r < h_r)
        food_right = float(gf_c > h_c)
        food_down  = float(gf_r > h_r)
        food_left  = float(gf_c < h_c)

        # [11] Comida mala delante
        bad_front = float(next_cell(head, front) == self.bad_food)

        # [12] Obstáculo delante (adicionalmente al peligro de colisión)
        obs_front = float(next_cell(head, front) in self.obstacles)

        # [13] Longitud normalizada (para que el agente conozca su tamaño)
        length_norm = len(self.snake) / (self.grid_size ** 2)

        obs = np.array([
            danger_front, danger_left, danger_right,
            *dir_one_hot,
            food_up, food_right, food_down, food_left,
            bad_front, obs_front,
            length_norm
        ], dtype=np.float32)

        return obs

    def _is_collision(self, pos) -> bool:
        """Devuelve True si la posición está fuera de la cuadrícula o es el cuerpo."""
        r, c = pos
        if r < 0 or r >= self.grid_size or c < 0 or c >= self.grid_size:
            return True
        # Colisión con el cuerpo (excluimos la cabeza actual)
        return pos in list(self.snake)

    def _distance_reward(self, head) -> float:
        """
        Recompensa de forma (shaping) basada en distancia Manhattan.
        +0.1 si nos acercamos, -0.1 si nos alejamos.
        Esto acelera el aprendizaje inicial guiando al agente sin
        alterar la política óptima (reward shaping).
        """
        gf_r, gf_c = self.good_food
        h_r, h_c = head
        dist = abs(gf_r - h_r) + abs(gf_c - h_c)

        # Comparamos con la distancia anterior (antes de mover)
        prev_r, prev_c = list(self.snake)[1] if len(self.snake) > 1 else head
        prev_dist = abs(gf_r - prev_r) + abs(gf_c - prev_c)

        if dist < prev_dist:
            return 0.1
        elif dist > prev_dist:
            return -0.1
        return 0.0

    def _free_cells(self) -> list:
        """Devuelve celdas libres (sin serpiente, sin comida, sin obstáculos)."""
        occupied = set(self.snake)
        if self.good_food:
            occupied.add(self.good_food)
        if self.bad_food:
            occupied.add(self.bad_food)
        if self.obstacles:
            occupied.update(self.obstacles)
        all_cells = {(r, c) for r in range(self.grid_size) for c in range(self.grid_size)}
        return list(all_cells - occupied)

    def _place_food(self):
        """Coloca comida buena y mala en posiciones libres aleatorias."""
        free = self._free_cells()
        if len(free) >= 2:
            positions = random.sample(free, 2)
            self.good_food = positions[0]
            self.bad_food  = positions[1]
        elif len(free) == 1:
            self.good_food = free[0]
            self.bad_food  = None
        # Si no hay celdas libres, no se coloca comida (episodio casi terminado)

    def _place_obstacles(self):
        """Coloca obstáculos en posiciones libres al inicio del episodio."""
        free = self._free_cells()
        n = min(self.n_obstacles, len(free))
        self.obstacles = set(random.sample(free, n))

    def _move_obstacles(self):
        """
        Mueve cada obstáculo a una celda adyacente libre aleatoriamente.
        Esto introduce la estocasticidad del entorno: P(s'|s,a) no es
        completamente determinista, lo que justifica el uso de DQN
        frente a métodos tabulares puros.
        """
        new_obstacles = set()
        for obs in self.obstacles:
            neighbors = []
            for dr, dc in self.DIRECTIONS:
                nr, nc = obs[0] + dr, obs[1] + dc
                candidate = (nr, nc)
                if (0 <= nr < self.grid_size and
                    0 <= nc < self.grid_size and
                    candidate not in self.snake and
                    candidate != self.good_food and
                    candidate != self.bad_food and
                    candidate not in new_obstacles):
                    neighbors.append(candidate)
            if neighbors:
                new_obstacles.add(random.choice(neighbors))
            else:
                new_obstacles.add(obs)  # se queda si no puede moverse
        self.obstacles = new_obstacles

In [ ]:
env = SnakeEnv(grid_size=10)
obs, info = env.reset()

total_reward = 0
for _ in range(200):
    action = env.action_space.sample()  # política aleatoria
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    env.render()
    if terminated or truncated:
        print(f"Episodio terminado. Recompensa total: {total_reward:.2f} | Causa: {info.get('cause')}")
        obs, info = env.reset()
        total_reward = 0

env.close()